<div style="
    font-size: 16px;
    font-weight: bold;
    background-color: #e0e0e0;
    padding: 6px;
    border-radius: 5px;
    margin: 10px 0;
">
    Here we load and clean the historical data of loan transactions. This data was sourced from Kaggle therefore it is anonymised. 
</div>

<div style="
    font-size: 14px;
    font-weight: bold;
    font-color: #e0e0e0;
    background-color: #e0e0e0;
    padding: 6px;
    border-radius: 5px;
    margin: 10px 0;
">
    Dataset:
</div> <div style="
    font-size: 14px;
    background-color: #e0e0e0;
    padding: 6px;
    border-radius: 5px;
    margin: 10px 0;
">
    LendingClub Loan Data (CSV) https://www.kaggle.com/datasets/adarshsng/lending-club-loan-data-csv?select=loan.csv
</div>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
use_cols = [
    "loan_status",
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "grade",
    "open_acc",
    "revol_util",
    "earliest_cr_line"
]
    
    

data = pd.read_csv(r"C:\Users\CASH\Downloads\loan.csv\loan.csv", usecols=use_cols, nrows=1000000)

</div> <div style="
    font-size: 16px;
    background-color: #e0e0e0;
    padding: 6px;
    border-radius: 5px;
    margin: 10px 0;
">
    A quick inspection of the data before we proceed.
</div>

In [3]:
data.head()

,loan_amnt,term,int_rate,installment,grade,annual_inc,loan_status,dti,earliest_cr_line,open_acc,revol_util
0,2500,36 months,13.56,84.92,C,55000.0,Current,18.24,Apr-2001,9,10.3
1,30000,60 months,18.94,777.23,D,90000.0,Current,26.52,Jun-1987,13,24.2
2,5000,36 months,17.97,180.69,D,59280.0,Current,10.51,Apr-2011,8,19.1
3,4000,36 months,18.94,146.51,D,92000.0,Current,16.74,Feb-2006,10,78.1
4,30000,60 months,16.14,731.78,C,57250.0,Current,26.35,Dec-2000,12,3.6


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 11 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   loan_amnt         1000000 non-null  int64  
 1   term              1000000 non-null  object 
 2   int_rate          1000000 non-null  float64
 3   installment       1000000 non-null  float64
 4   grade             1000000 non-null  object 
 5   annual_inc        1000000 non-null  float64
 6   loan_status       1000000 non-null  object 
 7   dti               998803 non-null   float64
 8   earliest_cr_line  1000000 non-null  object 
 9   open_acc          1000000 non-null  int64  
 10  revol_util        999113 non-null   float64
dtypes: float64(5), int64(2), object(4)
memory usage: 83.9+ MB


In [5]:
print(data["loan_status"].value_counts())

print("Here we see the statuses of the loans of the clients.")

loan_status
Current               597668
Fully Paid            297123
Charged Off            86058
Late (31-120 days)     11945
In Grace Period         5051
Late (16-30 days)       2134
Default                   21
Name: count, dtype: int64
Here we see the statuses of the loans of the clients.


</div> <div style="
    font-size: 14px;
    background-color: #e0e0e0;
    padding: 6px;
    border-radius: 5px;
    margin: 10px 0;
">
    The original dataset was very large and contained a lot of information that we do not need for this model so we have trimmed it down for better loading speed and cleaner reading. It initially had 145 columns, we have selected eleven of the ones we need. 5 of the 11 columns contains float entries, that is decimal point numbers. 2 of the columns have integer values and 4 of them are objects. We have also trimmed the dataset to just the first 1,000,000 entries to train and test our model with.
</div>

In [6]:
# We will create a new column Crediy History Years from the object column earliest_cr_line.

data["earliest_cr_line"] = pd.to_datetime(data["earliest_cr_line"], format="%b-%Y", errors="coerce")

data["credit_history_years"] = (pd.Timestamp("today") - data["earliest_cr_line"]).dt.days/365

data["credit_history_years"] = data["credit_history_years"].round(1)

In [7]:
# We first create a column of defaulted and paid off loans from clients
data = data[data["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()
data.loc[:, "default"] = data["loan_status"].map({
    "Fully Paid": 0,
    "Charged Off": 1
})


In [8]:
data[["credit_history_years", "default"]].head()
data["credit_history_years"].describe()

count    383181.000000
mean         25.935427
std           7.651856
min          10.400000
25%          20.800000
50%          24.400000
75%          29.800000
max          81.000000
Name: credit_history_years, dtype: float64

In [9]:
# The cell below produced a ValueError at first so we are making a minor change to fix that

data["term"] = (
    data["term"].astype(str).str.extract(r"(\d+)").astype(int))

<div style="
    font-size: 16px;
    font-weight: bold;
    background-color: #e0e0e0;
    padding: 6px;
    border-radius: 5px;
    margin: 10px 0;
">
    Now we will encode some features and also split the dataset into what we'll use to train and what we'll use to test the model. 
</div>

In [10]:
data = data.sample(n = min(20000, len(data)), random_state=42)

In [11]:
Y = data["default"].values
X = data.loc[:, data.columns != "default"]

In [12]:
X.shape, Y.shape # Confirmation

((20000, 12), (20000,))

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size = 0.3, # We use a 70/30 split
    random_state=42,
    stratify=Y
)

In [14]:
categorical_features = ["grade"]

numerical_features = [
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "open_acc",
    "revol_util",
    "credit_history_years"
]

<div style="
    font-size: 16px;
    font-weight: bold;
    background-color: #e0e0e0;
    padding: 6px;
    border-radius: 5px;
    margin: 10px 0;
">
    We begin building the Processing Pipeline.
</div>

In [15]:
# While later building the regression model we were met with a NaN value so in this cell we will implement code to handle that error

from sklearn.impute import SimpleImputer

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

</div> <div style="
    font-size: 14px;
    background-color: #e0e0e0;
    padding: 6px;
    border-radius: 5px;
    margin: 10px 0;
">
    We'll construct the linear regression model
</div>

In [17]:

from sklearn.linear_model import LogisticRegression

log_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter = 1000))
         ]
)

# Let's train it

log_model.fit(X_train, Y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['loan_amnt', 'term',
                                                   'int_rate', 'installment',
                                                   'annual_inc', 'dti',
                                                   'open_acc', 'revol_util',
                                                   'credit_history_years']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  ['grade'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

</div> <div style="
    font-size: 14px;
    background-color: #e0e0e0;
    padding: 6px;
    border-radius: 5px;
    margin: 10px 0;
">
    Now that we have our pipeline let's evaluate the model.
</div>

In [18]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

In [19]:
Y_pred = log_model.predict(X_test)
Y_proba = log_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(Y_test, Y_pred))
print("ROC-AUC:", roc_auc_score(Y_test, Y_proba))

Accuracy: 0.7778333333333334
ROC-AUC: 0.6883320888519758


In [20]:
report = classification_report(Y_test, Y_pred, output_dict = True)
df_report = pd.DataFrame(report).T

def highlight_low(val, threshold=0.6):
    return "background-color: #f8d7da" if val < threshold else ""

def highlight_low1(val, threshold=0.4):
    return "background-color: #f8d7da" if val < threshold else ""

def highlight(val, threshold=2000):
    return "background-color: #f8d7da" if 10 < val < threshold else ""

df_report.style \
    .format("{:.3f}") \
    .map(highlight, subset=["support"]) \
    .map(highlight_low, subset=["precision"]) \
    .map(highlight_low1, subset=["recall", "f1-score"]) \
    .set_caption("Classification Report") \
    .set_table_styles([
        {"selector": "table",
         "props": [("border", "2px solid black")]},
        {"selector": "th, td",
         "props": [("border", "1px solid black")]}
    ])




,precision,recall,f1-score,support
0,0.783,0.988,0.873,4655.000
1,0.547,0.052,0.095,1345.000
accuracy,0.778,0.778,0.778,0.778
macro avg,0.665,0.520,0.484,6000.000
weighted avg,0.730,0.778,0.699,6000.000


<div style="
    font-size: 14px;
    font-weight: bold;
    background-color: #e0e0e0;
    padding: 6px;
    border-radius: 5px;
    margin: 10px 0;
">
    Using our primary metric of choice which Receiver Operating Characteristic - Area Under the Curve. this measures how well the model separates defaulters from non-defaulters. The result from our model - ROC-AUC: 0.6883320888519758. Which is roughly 0.69. This means are model does a decent job of separating defaulters from non-defaulters. For referenc a score of 0.5 would mean our model is randomly guessing, a score of 0.8+ means the model is very good. Our model does an okay job landing in between 0.5 and 0.8 but closer to 0.8.
</div>